 **SQL Queries**

In [0]:
%sql

select * from ecommerce.events_delta

event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
2019-11-20T17:08:37.000Z,view,13100745,2053013553526341921,null,skad,177.1,49484535,16d460d5-d37e-4a4d-bb9c-3e9bb862dfe7
2019-11-24T15:22:13.000Z,view,2701022,2053013563911439225,appliances.kitchen.refrigerators,leadbros,198.2,116566414,895b1f64-399a-4f71-9acb-73888b57c243
2019-11-13T16:59:02.000Z,view,11700129,2053013554591695207,electronics.audio.acoustic,yamaha,236.81,154128341,37895eee-89df-482f-a1ce-730967c1bf63
2019-11-19T16:41:54.000Z,view,4502521,2053013563877884791,appliances.kitchen.hob,bosch,360.09,191365178,82907d05-cf00-42f4-b09b-49af25a3d69b
2019-11-10T15:46:41.000Z,view,22700959,2053013556168753601,null,hp,84.94,207295894,1d99255d-f576-4ef0-b71d-e91a09639ad3
2019-11-09T17:24:47.000Z,view,26201182,2053013563693335403,null,lucente,122.27,210089363,ad2f34c0-0d00-4511-be4f-bcd95b9d1da2
2019-11-25T15:38:14.000Z,view,2500263,2053013564003713919,appliances.kitchen.oven,bosch,334.6,268846425,2939e7d5-81f8-4748-968b-69911c4cbadf
2019-11-18T08:16:16.000Z,view,2800627,2053013563835941749,appliances.kitchen.refrigerators,leadbros,248.4,294504596,6d897c85-71fd-41ab-8e16-65b1556ff301
2019-11-18T10:01:46.000Z,view,31501161,2053013558031024687,null,luminarc,107.85,295708261,021808d9-c281-41e1-a4f7-ac624dec18d7
2019-11-27T18:35:56.000Z,view,5100773,2053013553341792533,electronics.clocks,acme,89.84,299358698,4c3b5a1d-c932-47ee-bad8-afe59fcc597a


Databricks visualization. Run in Databricks to view.

In [0]:

%sql
--distinct brands and event_type
select distinct event_type,brand from ecommerce.events_delta



event_type,brand
view,skad
cart,omron
view,kabrita
view,barer
view,ruixinlang
view,micio
cart,pandect
view,berber
view,babycare
cart,neumann


In [0]:
%sql
--Total sales
select round(sum(price),2) from ecommerce.events_delta

---------------------------------------------------------------------------
ParseException                            Traceback (most recent call last)
File <command-4979079017267941>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', '--Total sales\nselect round(sum(price),2) from ecommerce.events_delta\n\n--total sales by brand\nselect brand,round(sum(price),2) as sales from ecommerce.events_delta group by brand order by sales desc\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/

In [0]:
%sql
--total sales by brand
select brand,round(sum(price),2) as sales from ecommerce.events_delta group by brand order by sales desc

"coalesce(brand,""No brand"")",sales
apple,5.15784077353E9
samsung,2.80079141781E9
No brand,1.93712227627E9
xiaomi,9.4319340578E8
lg,5.1484253371E8
acer,4.384314986E8
lenovo,4.0702721614E8
huawei,3.715936696E8
sony,3.5229663469E8
asus,2.913340478E8


In [0]:
%sql
WITH daily_revenue AS (
    SELECT
        DATE(event_time) AS event_date,
        round(SUM(price)) AS total_sales
    FROM ecommerce.events_delta
    WHERE event_type = 'purchase'
    GROUP BY DATE(event_time)
)

SELECT
    event_date,
    total_sales,
    AVG(total_sales) OVER (
        ORDER BY event_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS moving_avg_7d
FROM daily_revenue
ORDER BY event_date;


event_date,total_sales,moving_avg_7d
2019-11-01,6949402.0,6949402.0
2019-11-02,6389578.0,6669490.0
2019-11-03,6656920.0,6665300.0
2019-11-04,8033900.0,7007450.0
2019-11-05,7248732.0,7055706.4
2019-11-06,7397761.0,7112715.5
2019-11-07,7061896.0,7105455.571428572
2019-11-08,7742040.0,7218689.571428572
2019-11-09,6646709.0,7255422.571428572
2019-11-10,6633475.0,7252073.285714285


Databricks visualization. Run in Databricks to view.

In [0]:
%sql


WITH funnel_steps AS (
    SELECT
        user_id,
        MAX(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS viewed,
        MAX(CASE WHEN event_type = 'add_to_cart' THEN 1 ELSE 0 END) AS added_to_cart,
        MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchased
    FROM ecommerce.events_delta
    GROUP BY user_id
)

SELECT
    COUNT(DISTINCT user_id) AS total_users,
    SUM(viewed) AS view_users,
    SUM(added_to_cart) AS add_to_cart_users,
    SUM(purchased) AS purchase_users,

    ROUND(
      100.0 * TRY_DIVIDE(SUM(added_to_cart), SUM(viewed)),
      2
    ) AS view_to_cart_conv_pct,

    ROUND(
      100.0 * TRY_DIVIDE(SUM(purchased), SUM(added_to_cart)),
      2
    ) AS cart_to_purchase_conv_pct
FROM funnel_steps;


total_users,view_users,add_to_cart_users,purchase_users,view_to_cart_conv_pct,cart_to_purchase_conv_pct
3696117,3695420,0,441637,0.0,null
